# Amazon ML Challenge 2026 — entity resolution
## Two T4 GPUs · grouped 70/30 validation · reproducible submission

**Read first:** the actual challenge TSVs have not been available to the notebook author. Local synthetic CPU tests verify software behavior, not real challenge accuracy or dual-T4 performance. This notebook measures your real result when executed on Kaggle. No winning score is promised.

**Setup:** attach your dataset, select **GPU T4 x2**, then run every cell in order. The configured path is the one you provided; a single alternate `student_resource` mount is discovered automatically. Enable internet only to install missing packages, or use attached package wheels. No business lookup, pretrained model download, geocoding or external training data is used.

**Output:** `/kaggle/working/amazon_er/output/matching_results.tsv` is the leaderboard file. The final ZIP contains both TSVs, models, runnable code and methodology. Read `reports/holdout_metrics.json` for the 30% evaluation.

## Architecture and validation

1. Audit TSVs, normalize text, keep numbers and full/core names.
2. Group shared business records; split approximately **70% training / 30% untouched holdout**.
3. Inside the 70%, use **56% overall for inner fitting / 14% for calibration**.
4. Retrieve candidates separately from S2 and S3 using four lexical branches and exact signatures.
5. Build 50 numerical pair features; train two CatBoost seed models using both T4s per model.
6. Select iterations and threshold only inside the 70%; refit on all 70%; report the 30% result.
7. Refit on all supplied training labels, predict test, validate both outputs and package.

**Corrections to the proposed design:** multiple matches are allowed; country labels are open-ended; the threshold is calibrated rather than fixed to 0.78. Country blocking is a configurable assumption checked on inner-fit positives. Character hashing supports unseen substrings; this version does not use a semantic encoder.

**Main metric:** macro F0.5 per S1, including singletons. Exact-set accuracy, precision, recall, candidate recall and singleton accuracy are also reported. Candidate-pair accuracy is diagnostic only. The holdout cannot measure unseen France performance.

## Cell 1 — Environment and output directory

In [ ]:
from pathlib import Path
import os,sys,subprocess,importlib.metadata
BASE=Path('/kaggle/working/amazon_er')
(BASE/'src').mkdir(parents=True,exist_ok=True)
for name in ['artifacts','cache','models','reports','output']:
    (BASE/name).mkdir(exist_ok=True)
os.chdir(BASE)

# Optional offline wheel folder, e.g. /kaggle/input/my-python-wheels/wheels
WHEEL_DIR=None
pins={'catboost':'1.2.8','rapidfuzz':'3.13.0','sparse-dot-topn':'1.1.5'}
needed=[]
for package,version in pins.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: needed.append(f'{package}=={version}')
if needed:
    command=[sys.executable,'-m','pip','install','--disable-pip-version-check','--quiet']
    if WHEEL_DIR: command+=['--no-index','--find-links',str(WHEEL_DIR)]
    subprocess.run(command+needed,check=True)
# Kaggle supplies these base scientific packages. Keep its compatible CUDA/runtime stack.
import numpy,pandas,scipy,sklearn,joblib,catboost,rapidfuzz,sparse_dot_topn
print('Python:',sys.version)
print('Packages:', {p:importlib.metadata.version(p) for p in pins})
print('Working directory:',BASE)
try:
    subprocess.run(['nvidia-smi','-L'],check=True)
except (FileNotFoundError,subprocess.CalledProcessError):
    print('No NVIDIA GPUs detected. Enable GPU T4 x2; CPU fallback is slower.')


## Cell 2 — Configuration
Change `TEAM_NAME` if desired. Defaults request both T4 GPUs. After changing settings, rerun preparation and every later stage.

In [ ]:
from pathlib import Path
import json
BASE=Path('/kaggle/working/amazon_er')
DATA_ROOT='/kaggle/input/datasets/mayankr293/aws-ml-challenge-dataset/student_resource'
TEAM_NAME='amazon_ml_team'
settings={
    'data_root':DATA_ROOT,'work_dir':str(BASE),'team_name':TEAM_NAME,
    'seed':2026,'holdout_fraction':0.30,'calibration_fraction_of_training':0.20,
    'country_block':'auto',
    'task_type':'GPU','devices':'0:1','allow_cpu_fallback':True,'gpu_ram_part':0.75,
    'query_batch_size':256,'transform_batch_size':12000,
    'vectorizer_fit_records':180000,'max_train_pairs':4000000,
    'name_topk':16,'address_topk':10,'word_topk':12,'hash_topk':6,
    'similarity_floor':0.05,
    'hard_negatives_per_entity':10,'random_negatives_per_entity':3,
    'max_iterations':1400,'early_stopping_rounds':100,
    'learning_rate':0.055,'depth':7,'l2_leaf_reg':6.0,'seeds':[2026,3407],
    'keep_training_cache':False,'run_official_full_candidate_check':False,
}
(BASE/'run_config.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
print(json.dumps(settings,indent=2))


## Cell 3 — Configuration and normalization
This cell writes `src/common.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/common.py
"""Shared configuration and conservative, Unicode-aware text processing."""
from pathlib import Path
import hashlib, json, os, re, time, unicodedata
import numpy as np

DEFAULTS = dict(
    data_root='/kaggle/input/datasets/mayankr293/aws-ml-challenge-dataset/student_resource',
    work_dir='/kaggle/working/amazon_er', team_name='amazon_ml_team', seed=2026,
    holdout_fraction=0.30, calibration_fraction_of_training=0.20,
    country_block='auto', query_batch_size=256, transform_batch_size=12000,
    threads=max(1, min(8, os.cpu_count() or 2)),
    vectorizer_fit_records=180000, name_max_features=140000,
    address_max_features=180000, word_max_features=160000,
    name_topk=16, address_topk=10, word_topk=12, hash_topk=6,
    similarity_floor=0.05, max_train_pairs=4000000,
    hard_negatives_per_entity=10, random_negatives_per_entity=3,
    max_iterations=1400, learning_rate=0.055, depth=7, l2_leaf_reg=6.0,
    early_stopping_rounds=100, task_type='GPU', devices='0:1',
    allow_cpu_fallback=True, gpu_ram_part=0.75, seeds=[2026, 3407],
    threshold_grid=[round(float(x), 4) for x in np.r_[np.linspace(.05,.95,91), .97,.98,.99,.995]]+[1.000001],
    keep_training_cache=False, run_official_full_candidate_check=False,
)

def log(message):
    print(time.strftime('%H:%M:%S'), message, flush=True)

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def config(path, overrides=None):
    cfg = DEFAULTS | json.loads(Path(path).read_text()) | (overrides or {})
    cfg['work_dir'] = str(Path(cfg['work_dir']).resolve())
    for folder in ['artifacts','cache','reports','output','models']:
        (Path(cfg['work_dir']) / folder).mkdir(parents=True, exist_ok=True)
    return cfg

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024), b''): h.update(b)
    return h.hexdigest()

def hash_fraction(text, seed):
    return int.from_bytes(hashlib.blake2b(f'{seed}|{text}'.encode(),digest_size=8).digest(),'big')/2**64

def normalize(text):
    text = unicodedata.normalize('NFKD',str(text).casefold())
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.replace('&',' and ')
    return ' '.join(re.sub(r'[^\w\s]',' ',text,flags=re.UNICODE).replace('_',' ').split())

# Handwritten spelling conventions, not an external address or business database.
NAME_WORDS = {'corporation':'corp','incorporated':'inc','limited':'ltd',
              'private':'pvt','company':'co'}
ROAD_WORDS = {'road':'rd','street':'st','avenue':'ave','boulevard':'blvd',
              'building':'bldg','apartment':'apt','suite':'ste','floor':'fl',
              'highway':'hwy','lane':'ln','sector':'sec','number':'no'}
LEGAL = set('corp inc ltd pvt co llc llp plc sa sas sarl eurl'.split())

def name_text(text):
    return ' '.join(NAME_WORDS.get(w,w) for w in normalize(text).split())

def core_name(text):
    words=text.split()
    while len(words)>1 and words[-1] in LEGAL: words.pop()
    return ' '.join(words)

def address_text(text):
    return ' '.join(ROAD_WORDS.get(w,w) for w in normalize(text).split())

def country_text(text):
    value=normalize(text)
    # Unrecognized countries are preserved; no fixed country vocabulary.
    return {'united states':'us','united states of america':'us','usa':'us',
            'in':'india','ind':'india','fr':'france'}.get(value,value)

def source_paths(root, split):
    root=Path(root)
    return [root/'dataset'/split/f'{split}_source{i}.tsv' for i in [1,2,3]]


## Cell 4 — Input audit and entity-group splits
This cell writes `src/data.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/data.py
"""Read TSVs and isolate real-entity groups before making the 70/30 split."""
from pathlib import Path
import gc, json, shutil
import numpy as np
import pandas as pd
from common import *

REQUIRED=['entity_id','business_name','business_address','country']

def resolve_root(preferred):
    p=Path(preferred)
    if (p/'dataset/train/train_ground_truth.tsv').exists(): return p
    hits=list(Path('/kaggle/input').glob('**/student_resource/dataset/train/train_ground_truth.tsv')) if Path('/kaggle/input').exists() else []
    if len(hits)==1: return hits[0].parents[2]
    raise FileNotFoundError(f'Dataset root not found: {p}. Found {len(hits)} alternatives. Set DATA_ROOT in the configuration cell.')

def read_source(path, prefix):
    frame=pd.read_csv(path,sep='\t',dtype=str,keep_default_na=False)
    if not set(REQUIRED)<=set(frame): raise ValueError(f'{path}: expected {REQUIRED}, got {list(frame)}')
    frame=frame[REQUIRED]
    if frame.entity_id.duplicated().any() or not frame.entity_id.str.startswith(prefix+'-').all():
        raise ValueError(f'Invalid/duplicate {prefix} IDs in {path}')
    frame['name']=frame.business_name.map(name_text)
    frame['core']=frame.name.map(core_name)
    frame['address']=frame.business_address.map(address_text)
    frame['country']=frame.country.map(country_text)
    frame['source']=int(prefix[-1])
    return frame[['entity_id','name','core','address','country','source']]

def read_split(root, split):
    parts=[]
    for i,p in enumerate(source_paths(root,split),start=1):
        log(f'Reading {p.name}')
        parts.append(read_source(p,f'S{i}'))
    return parts[0].reset_index(drop=True),pd.concat(parts[1:],ignore_index=True)

def read_truth(root, queries, targets):
    p=Path(root)/'dataset/train/train_ground_truth.tsv'
    df=pd.read_csv(p,sep='\t',dtype=str,keep_default_na=False)
    if list(df.columns)!=['source1_entity_id','matched_entity_ids']: raise ValueError(f'Unexpected ground truth header: {list(df)}')
    if df.source1_entity_id.duplicated().any(): raise ValueError('Duplicate S1 ground truth rows')
    if set(df.source1_entity_id)!=set(queries.entity_id): raise ValueError('Ground truth must contain every training S1 exactly once')
    target_ids=set(targets.entity_id)
    truth={}
    for sid,text in df.itertuples(index=False,name=None):
        ids=[x.strip() for x in text.split(',') if x.strip()]
        if len(ids)!=len(set(ids)) or any(x not in target_ids for x in ids): raise ValueError(f'Invalid ground truth IDs for {sid}')
        truth[sid]=frozenset(ids)
    return truth

class DSU:
    def __init__(self,n): self.parent=np.arange(n)
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return int(x)
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b:self.parent[b]=a

def make_groups(q,t,truth):
    """Shared target IDs and identical nonempty records cannot cross splits."""
    dsu=DSU(len(q)); owner={}
    for i,sid in enumerate(q.entity_id):
        for tid in truth[sid]:
            if tid in owner: dsu.union(i,owner[tid])
            else: owner[tid]=i
    # Connect duplicate normalized signatures across ALL labeled entity records.
    seen={}
    for i,row in enumerate(q.itertuples(index=False)):
        if row.name and row.address:
            key=(row.name,row.address,row.country)
            if key in seen:dsu.union(i,seen[key])
            else:seen[key]=i
    for row in t.itertuples(index=False):
        if row.entity_id in owner and row.name and row.address:
            i=owner[row.entity_id]; key=(row.name,row.address,row.country)
            if key in seen:dsu.union(i,seen[key])
            else:seen[key]=i
    groups=np.array([dsu.find(i) for i in range(len(q))])
    return groups,owner,seen

def split_groups(indices, groups, q, truth, fraction, seed):
    """Stratify groups by country and singleton status where strata permit it."""
    from sklearn.model_selection import train_test_split
    representatives={}
    for i in indices:representatives.setdefault(int(groups[i]),int(i))
    keys=np.array(sorted(representatives))
    if len(keys)<4:raise ValueError('Need at least four independent entity groups for this split.')
    labels=[q.iloc[representatives[k]].country+'|'+str(bool(truth[q.iloc[representatives[k]].entity_id])) for k in keys]
    counts=pd.Series(labels).value_counts()
    ntest=int(np.ceil(len(keys)*fraction))
    stratify=labels if counts.min()>=2 and ntest>=len(counts) and len(keys)-ntest>=len(counts) else None
    a,b=train_test_split(keys,test_size=fraction,random_state=seed,stratify=stratify)
    return np.array(indices)[np.isin(groups[indices],a)],np.array(indices)[np.isin(groups[indices],b)]

def assign_splits(q,t,truth,cfg):
    groups,owner,seen=make_groups(q,t,truth)
    dev,hold=split_groups(np.arange(len(q)),groups,q,truth,cfg['holdout_fraction'],cfg['seed'])
    fit,cal=split_groups(dev,groups,q,truth,cfg['calibration_fraction_of_training'],cfg['seed']+1)
    tag=np.full(len(q),'holdout',dtype=object);tag[fit]='fit';tag[cal]='calibration'
    group_tag={int(groups[i]):tag[i] for i in range(len(q))}
    target_tags=[]
    for row in t.itertuples(index=False):
        # Unlinked duplicate target rows follow the signature's group, too.
        oi=owner.get(row.entity_id,seen.get((row.name,row.address,row.country)) if row.name and row.address else None)
        if oi is not None:part=group_tag[int(groups[oi])]
        else:
            key='|'.join([row.name,row.address,row.country]) if row.name or row.address else row.entity_id
            value=hash_fraction(key,cfg['seed'])
            h=cfg['holdout_fraction'];c=(1-h)*cfg['calibration_fraction_of_training']
            part='holdout' if value<h else ('calibration' if value<h+c else 'fit')
        target_tags.append(part)
    q=q.copy();t=t.copy();q['split']=tag;t['split']=target_tags
    lookup=dict(zip(t.entity_id,t.split))
    for sid,part in zip(q.entity_id,q.split):
        assert all(lookup[x]==part for x in truth[sid]),'Entity leaked across partitions'
    q['group_id']=groups
    return q,t

def prepare(cfg):
    work=Path(cfg['work_dir']); root=resolve_root(cfg['data_root'])
    cfg['data_root']=str(root)
    q,t=read_split(root,'train');truth=read_truth(root,q,t)
    q,t=assign_splits(q,t,truth,cfg)
    country_lookup=dict(zip(t.entity_id,t.country))
    cross=sum(1 for row in q[q.split=='fit'].itertuples(index=False) for tid in truth[row.entity_id]
              if row.country and country_lookup[tid] and row.country!=country_lookup[tid])
    cfg['use_country_block']=(cross==0) if cfg['country_block']=='auto' else bool(cfg['country_block'])
    if cfg['use_country_block'] and cross:raise ValueError('Country blocking excludes labeled positives in inner training; use country_block=false.')
    q.to_pickle(work/'cache/train_queries.pkl');t.to_pickle(work/'cache/train_targets.pkl')
    write_json(work/'artifacts/truth.json',{k:sorted(v) for k,v in truth.items()})
    q[['entity_id','group_id','split','country']].to_csv(work/'reports/entity_split.tsv',sep='\t',index=False)
    report={'query_counts':q.split.value_counts().to_dict(),'target_counts':t.split.value_counts().to_dict(),
            'actual_outer_train_fraction':float((q.split!='holdout').mean()),
            'actual_outer_holdout_fraction':float((q.split=='holdout').mean()),
            'singleton_fraction':float(np.mean([not truth[x] for x in q.entity_id])),
            'country_block_enabled':cfg['use_country_block'],'cross_country_inner_fit_links':cross,
            'country_block_assumption':'Country labels are consistent across matching records; unknown countries search globally. Unseen country names remain valid.',
            'split_note':'70/30 by complete entity groups, approximately 70/30 by rows if some groups contain several S1 records.'}
    manifest=[]
    for split in ['train','test']:
        for p in source_paths(root,split):
            if not p.exists():raise FileNotFoundError(p)
            counts={};n=0
            for c in pd.read_csv(p,sep='\t',dtype=str,keep_default_na=False,chunksize=150000):
                if not set(REQUIRED)<=set(c):raise ValueError(f'Invalid schema: {p}')
                n+=len(c)
                for k,v in c.country.map(country_text).value_counts().items():counts[k]=counts.get(k,0)+int(v)
            manifest.append({'file':str(p.relative_to(root)),'rows':n,'countries':counts,'bytes':p.stat().st_size,'sha256':sha256(p)})
    gtpath=root/'dataset/train/train_ground_truth.tsv'
    manifest.append({'file':str(gtpath.relative_to(root)),'sha256':sha256(gtpath),'bytes':gtpath.stat().st_size})
    report['files']=manifest
    for name in ['Documentation_template.md','README.md']:
        if (root/name).exists():shutil.copy2(root/name,work/'artifacts'/('provided_'+name))
    if (root/'utils/validate_submission.py').exists():
        shutil.copy2(root/'utils/validate_submission.py',work/'artifacts/official_validate_submission.py')
    else:raise FileNotFoundError('The provided utils/validate_submission.py is required.')
    write_json(work/'artifacts/resolved_config.json',cfg);write_json(work/'reports/data_audit.json',report)
    log(json.dumps({k:v for k,v in report.items() if k!='files'},indent=2))

def load_training(cfg, subset):
    work=Path(cfg['work_dir'])
    q=pd.read_pickle(work/'cache/train_queries.pkl');t=pd.read_pickle(work/'cache/train_targets.pkl')
    parts={'fit':['fit'],'calibration':['calibration'],'development':['fit','calibration'],
           'holdout':['holdout'],'all':['fit','calibration','holdout']}[subset]
    q=q[q.split.isin(parts)].reset_index(drop=True);t=t[t.split.isin(parts)].reset_index(drop=True)
    raw=json.loads((work/'artifacts/truth.json').read_text())
    truth={x:frozenset(raw[x]) for x in q.entity_id}
    return q,t,truth


## Cell 5 — Sparse candidate generation
This cell writes `src/retrieval.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/retrieval.py
"""Bounded sparse top-k search; exact duplicate signatures retain all copies."""
from pathlib import Path
import gc, json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sparse_dot_topn import sp_matmul_topn
from common import log

CHANNELS=['name','address','word','hash']

def documents(frame,channel):
    if channel in ['name','hash']:return frame['core'].to_numpy()
    if channel=='address':return frame['address'].to_numpy()
    return (frame['core']+' '+frame['address']).to_numpy()

def fit_vectorizers(q,t,cfg):
    rng=np.random.default_rng(cfg['seed']);frames=[]
    # Fit text statistics exclusively on inner-fit records (about 56% of S1).
    for f in [q,t]:
        n=min(len(f),cfg['vectorizer_fit_records']//2)
        frames.append(f.iloc[np.sort(rng.choice(len(f),n,replace=False))])
    sample=pd.concat(frames,ignore_index=True)
    models={}
    for channel in CHANNELS:
        if channel=='hash':
            # No vocabulary: unseen French substrings still have features.
            v=HashingVectorizer(analyzer='char_wb',ngram_range=(3,4),n_features=2**18,
                               alternate_sign=False,norm='l2',lowercase=False,dtype=np.float32)
        else:
            kw=dict(dtype=np.float32,lowercase=False,sublinear_tf=True,norm='l2',min_df=1,
                    max_features=cfg[{'name':'name_max_features','address':'address_max_features','word':'word_max_features'}[channel]])
            v=TfidfVectorizer(analyzer='word',ngram_range=(1,2),token_pattern=r'(?u)\b\w+\b',**kw) if channel=='word' else TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),**kw)
            texts=documents(sample,channel)
            if not any(str(x).strip() for x in texts):texts=['placeholder']
            v.fit(texts)
        models[channel]=v
        log(f'Fitted {channel} vectorizer')
    joblib.dump(models,Path(cfg['work_dir'])/'models/vectorizers.joblib',compress=3)
    return models

def transform_chunks(v,texts,chunk_size):
    if len(texts)==0:return v.transform([])
    chunks=[v.transform(texts[i:i+chunk_size]).tocsr() for i in range(0,len(texts),chunk_size)]
    return sparse.vstack(chunks,format='csr',dtype=np.float32)

def signatures(f):
    # Hash collisions create extra candidates, never automatic matches.
    return pd.util.hash_pandas_object(f[['name','address']],index=False).to_numpy(dtype=np.uint64)

class SourceIndex:
    def __init__(self,t,positions,models,cfg):
        self.positions=np.asarray(positions,dtype=np.int64);self.cfg=cfg;self.models=models
        rows=t.iloc[self.positions]
        self.B={};self.BT={}
        for c,v in models.items():
            self.B[c]=transform_chunks(v,documents(rows,c),cfg['transform_batch_size'])
            self.BT[c]=self.B[c].T.tocsr()
        h=signatures(rows);valid=((rows.name!='')&(rows.address!='')).to_numpy()
        order=np.argsort(h[valid],kind='stable');self.exact_rows=np.flatnonzero(valid)[order]
        self.exact_hash=h[self.exact_rows]
        nbytes=sum(x.data.nbytes+x.indices.nbytes+x.indptr.nbytes for x in [*self.B.values(),*self.BT.values()])
        log(f'Index ready: {len(rows):,} target records, sparse storage {nbytes/1024**3:.2f} GiB')

    def search(self,q):
        all_rows=[set() for _ in range(len(q))];Q={}
        for c,v in self.models.items():
            Q[c]=v.transform(documents(q,c)).tocsr()
            k=min(self.cfg[c+'_topk'],len(self.positions))
            if not k:continue
            C=sp_matmul_topn(Q[c],self.BT[c],top_n=k,threshold=self.cfg['similarity_floor'],
                            sort=True,n_threads=self.cfg['threads'])
            for i in range(len(q)):all_rows[i].update(C.indices[C.indptr[i]:C.indptr[i+1]])
        for i,h in enumerate(signatures(q)):
            if not q.iloc[i]['name'] or not q.iloc[i]['address']:continue
            lo=np.searchsorted(self.exact_hash,h,'left');hi=np.searchsorted(self.exact_hash,h,'right')
            all_rows[i].update(self.exact_rows[lo:hi])
        qi=[];ti=[]
        for i,rows in enumerate(all_rows):
            s=sorted(rows);qi.extend([i]*len(s));ti.extend(s)
        qi=np.array(qi,dtype=np.int32);ti=np.array(ti,dtype=np.int32)
        sim=np.zeros((len(qi),len(CHANNELS)),dtype=np.float32)
        # Compute each similarity for every candidate, even if a different channel retrieved it.
        for cidx,c in enumerate(CHANNELS):
            for start in range(0,len(qi),6000):
                sl=slice(start,start+6000)
                sim[sl,cidx]=np.asarray(Q[c][qi[sl]].multiply(self.B[c][ti[sl]]).sum(axis=1)).ravel()
        return qi,self.positions[ti],sim

def candidate_batches(q,t,cfg,models):
    """Each batch includes all queries, including zero-candidate queries."""
    empty=(q.name=='')&(q.address=='')
    # Avoid building an enormous global index for completely empty records.
    for start in range(0,int(empty.sum()),cfg['query_batch_size']):
        blank=q[empty].iloc[start:start+cfg['query_batch_size']].reset_index(drop=True)
        yield blank,np.empty(0,np.int32),np.empty(0,np.int64),np.empty((0,4),np.float32)
    q=q[~empty].reset_index(drop=True)
    groups=q.groupby('country',sort=True).indices if cfg['use_country_block'] else {'ALL':np.arange(len(q))}
    for country,qpositions in groups.items():
        allowed=((t.country==country)|(t.country=='')) if cfg['use_country_block'] and country else np.ones(len(t),bool)
        indexes=[]
        log(f'Preparing search for country={country!r}; queries={len(qpositions):,}')
        for src in [2,3]:
            pos=np.flatnonzero(np.asarray(allowed)&(t.source.to_numpy()==src))
            if len(pos):indexes.append(SourceIndex(t,pos,models,cfg))
        for start in range(0,len(qpositions),cfg['query_batch_size']):
            global_q=np.asarray(qpositions[start:start+cfg['query_batch_size']])
            b=q.iloc[global_q].reset_index(drop=True)
            results=[idx.search(b) for idx in indexes]
            if results:
                qi=np.concatenate([x[0] for x in results]);ti=np.concatenate([x[1] for x in results]);sim=np.vstack([x[2] for x in results])
                order=np.lexsort((ti,qi));qi,ti,sim=qi[order],ti[order],sim[order]
            else:qi=np.empty(0,np.int32);ti=np.empty(0,np.int64);sim=np.empty((0,4),np.float32)
            yield b,qi,ti,sim
            if start==0 or (start//cfg['query_batch_size'])%40==0:
                log(f'{country!r}: searched {min(start+len(b),len(qpositions)):,}/{len(qpositions):,} queries')
        indexes.clear();gc.collect()


## Cell 6 — Pair feature engineering
This cell writes `src/features.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/features.py
"""Numerical pair features only: IDs and raw country categories never enter ML."""
import re
import numpy as np
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein,JaroWinkler

FUZZ=[('ratio',fuzz.ratio,100.),('token_sort',fuzz.token_sort_ratio,100.),
      ('token_set',fuzz.token_set_ratio,100.),('levenshtein',Levenshtein.normalized_similarity,1.),
      ('jaro_winkler',JaroWinkler.normalized_similarity,1.)]
FEATURE_NAMES=['name_tfidf','address_tfidf','joint_word_tfidf','name_hash_cosine']
for field in ['name','core','address']:
    FEATURE_NAMES += [field+'_'+x[0] for x in FUZZ]
    FEATURE_NAMES += [field+'_'+x for x in ['exact','both_present','one_missing','length_ratio','token_jaccard','token_containment']]
FEATURE_NAMES += ['name_acronym_agree','number_jaccard','number_containment','number_disjoint',
                  'long_number_agree','long_number_conflict','first_number_agree','first_number_conflict',
                  'name_number_agree','name_number_conflict','country_agree','country_missing','target_is_s3']

def overlap(a,b):
    if not a or not b:return (0.,0.)
    inter=len(a&b)
    return inter/len(a|b),inter/min(len(a),len(b))

def metadata(frame):
    out=[]
    for row in frame.itertuples(index=False):
        nums=re.findall(r'\b\d+[a-z]*\b',row.address)
        out.append(dict(tokens={f:set(getattr(row,f).split()) for f in ['name','core','address']},
                        nums=set(nums),long=set(x for x in nums if len(x)>=4),first=nums[0] if nums else '',
                        nnums=set(re.findall(r'\b\d+\b',row.name)),
                        acronym=''.join(w[0] for w in row.core.split() if w and w not in ['and','the']),
                        compact=row.core.replace(' ','')))
    return out

def build_features(q,t,qi,ti,similarities,threads=2):
    if not len(qi):return np.empty((0,len(FEATURE_NAMES)),dtype=np.float32)
    unique,inv=np.unique(ti,return_inverse=True);tr=t.iloc[unique].reset_index(drop=True)
    qm=metadata(q);tm=metadata(tr)
    arrays=[similarities]
    for field in ['name','core','address']:
        left=q[field].to_numpy()[qi];right=tr[field].to_numpy()[inv]
        present=np.array([bool(a) and bool(b) for a,b in zip(left,right)])
        for _,scorer,scale in FUZZ:
            scores=process.cpdist(left,right,scorer=scorer,workers=threads,dtype=np.float32)/scale
            scores[~present]=0.;arrays.append(scores[:,None])
        values=[]
        for a,b,ix,jx in zip(left,right,qi,inv):
            ja,co=overlap(qm[ix]['tokens'][field],tm[jx]['tokens'][field])
            values.append([float(bool(a) and a==b),float(bool(a) and bool(b)),float(bool(a)!=bool(b)),
                           min(len(a),len(b))/max(1,len(a),len(b)),ja,co])
        arrays.append(np.asarray(values,dtype=np.float32))
    values=[];qc=q.country.to_numpy();tc=tr.country.to_numpy();ts=tr.source.to_numpy()
    for i,j in zip(qi,inv):
        a,b=qm[i],tm[j];ja,co=overlap(a['nums'],b['nums'])
        both=lambda key:bool(a[key]) and bool(b[key])
        values.append([
            float((len(a['acronym'])>=2 and a['acronym']==b['compact']) or (len(b['acronym'])>=2 and b['acronym']==a['compact'])),
            ja,co,float(both('nums') and not a['nums']&b['nums']),
            float(bool(a['long']&b['long'])),float(both('long') and not a['long']&b['long']),
            float(both('first') and a['first']==b['first']),float(both('first') and a['first']!=b['first']),
            float(bool(a['nnums']&b['nnums'])),float(both('nnums') and not a['nnums']&b['nnums']),
            float(bool(qc[i]) and qc[i]==tc[j]),float(not qc[i] or not tc[j]),float(ts[j]==3)])
    arrays.append(np.asarray(values,dtype=np.float32))
    X=np.hstack(arrays).astype(np.float32,copy=False)
    assert X.shape[1]==len(FEATURE_NAMES) and np.isfinite(X).all()
    return X

def pair_labels(q,t,qi,ti,truth):
    sids=q.entity_id.to_numpy();tids=t.entity_id.to_numpy()
    return np.fromiter((tids[j] in truth[sids[i]] for i,j in zip(qi,ti)),dtype=np.int8,count=len(qi))


## Cell 7 — Official metric and accuracy diagnostics
This cell writes `src/metrics.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/metrics.py
"""Competition macro F0.5, including singletons and blocking misses."""
import numpy as np

def entity_scores(truth_count, predicted_count, true_positive):
    p=np.asarray(predicted_count);tp=np.asarray(true_positive)
    den=4*p+truth_count
    return np.divide(5.*tp,den,out=np.ones_like(tp,dtype=float),where=den!=0)

class MetricAccumulator:
    def __init__(self,thresholds):
        self.thresholds=np.asarray(thresholds,dtype=float);self.groups={}
    def add(self,country,truth_count,labels,probabilities):
        order=np.argsort(-probabilities,kind='stable');p=probabilities[order];y=labels[order]
        n=np.searchsorted(-p,-self.thresholds,side='right')
        tp=np.r_[0,np.cumsum(y,dtype=np.int64)][n]
        f=entity_scores(truth_count,n,tp);exact=(n==truth_count)&(tp==truth_count)
        for group in ['ALL','country:'+country]:
            if group not in self.groups:
                self.groups[group]=dict(entities=0,truth_links=0,candidate_links=0,candidates=0,
                    matched_entities=0,singletons=0,f=np.zeros(len(n)),exact=np.zeros(len(n)),
                    pred=np.zeros(len(n)),tp=np.zeros(len(n)),empty_correct=np.zeros(len(n)),
                    blocking_macro_recall_sum=0.)
            a=self.groups[group];a['entities']+=1;a['truth_links']+=truth_count
            a['candidate_links']+=int(y.sum());a['candidates']+=len(y)
            a['matched_entities']+=int(truth_count>0);a['singletons']+=int(truth_count==0)
            a['f']+=f;a['exact']+=exact;a['pred']+=n;a['tp']+=tp
            a['empty_correct']+=(truth_count==0)&(n==0)
            a['blocking_macro_recall_sum']+=float(y.sum()/truth_count) if truth_count else 0.
        return n,tp,f
    def records(self,group='ALL'):
        a=self.groups[group];out=[]
        for i,threshold in enumerate(self.thresholds):
            tp=float(a['tp'][i]);pred=float(a['pred'][i]);true=a['truth_links']
            fp=pred-tp;fn=true-tp;tn=a['candidates']-a['candidate_links']-fp
            out.append(dict(threshold=float(threshold),entities=a['entities'],
                macro_f05=float(a['f'][i]/max(1,a['entities'])),
                entity_exact_match_accuracy=float(a['exact'][i]/max(1,a['entities'])),
                micro_precision=tp/pred if pred else 0.,micro_recall=tp/true if true else 0.,
                micro_f05=5*tp/(4*pred+true) if 4*pred+true else 1.,
                singleton_accuracy=float(a['empty_correct'][i]/a['singletons']) if a['singletons'] else None,
                candidate_recall=a['candidate_links']/true if true else None,
                candidate_macro_recall=a['blocking_macro_recall_sum']/a['matched_entities'] if a['matched_entities'] else None,
                candidate_pair_accuracy=(tp+tn)/a['candidates'] if a['candidates'] else None,
                true_positive=int(tp),false_positive=int(fp),false_negative=int(fn),
                true_links=int(true),candidate_links=int(a['candidate_links']),candidate_pairs=int(a['candidates']),
                predicted_links=int(pred),singletons=int(a['singletons'])))
        return out


## Cell 8 — Training, calibration and evaluation
This cell writes `src/training.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/training.py
"""Disk-backed pair features, GPU CatBoost ensemble and honest evaluation."""
from pathlib import Path
import csv,gc,json,shutil
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier,Pool,CatBoostError
from catboost.utils import get_gpu_device_count
from common import log,write_json
from features import FEATURE_NAMES,build_features,pair_labels
from retrieval import candidate_batches
from metrics import MetricAccumulator

def build_cache(q,t,truth,cfg,models,label):
    path=Path(cfg['work_dir'])/'cache'/label
    if path.exists():shutil.rmtree(path)
    path.mkdir();ntrue=sum(len(truth[s]) for s in q.entity_id)
    requested=cfg['hard_negatives_per_entity']+cfg['random_negatives_per_entity']
    budget=min(requested,max(0,(cfg['max_train_pairs']-ntrue)//max(1,len(q))))
    if budget<1:raise ValueError('max_train_pairs leaves no room for negatives. Increase it in configuration.')
    rng=np.random.default_rng(cfg['seed']);n=pos=ncandidate=recovered=0
    observed_queries=0;empty_queries=0
    with open(path/'X.dat','wb') as fx,open(path/'y.dat','wb') as fy,open(path/'w.dat','wb') as fw:
        for b,qi,ti,sim in candidate_batches(q,t,cfg,models):
            y=pair_labels(b,t,qi,ti,truth);keep=[];weights=[]
            ncandidate+=len(y);recovered+=int(y.sum());observed_queries+=len(b)
            starts=np.searchsorted(qi,np.arange(len(b)+1))
            for i in range(len(b)):
                ix=np.arange(starts[i],starts[i+1]);positive=ix[y[ix]==1];negative=ix[y[ix]==0]
                if not len(ix):empty_queries+=1;continue
                affinity=sim[negative] @ np.array([.4,.2,.25,.15],np.float32)
                hard_n=min(len(negative),budget,max(1,int(budget*.75)))
                hard=negative[np.argsort(-affinity,kind='stable')[:hard_n]]
                rest=np.setdiff1d(negative,hard,assume_unique=True)
                extra=rng.choice(rest,min(budget-len(hard),len(rest)),replace=False)
                selected=np.concatenate([positive,hard,extra]);selected.sort()
                keep.extend(selected);weights.extend([1./len(selected)]*len(selected))
            keep=np.asarray(keep,dtype=np.int64)
            X=build_features(b,t,qi[keep],ti[keep],sim[keep],cfg['threads'])
            X.tofile(fx);y[keep].astype(np.int8).tofile(fy);np.asarray(weights,np.float32).tofile(fw)
            n+=len(keep);pos+=int(y[keep].sum())
    if not n or not 0<pos<n:raise ValueError(f'{label}: training needs both labels. rows={n}, positives={pos}. Inspect input and retrieval.')
    assert n<=cfg['max_train_pairs']
    info=dict(rows=n,features=len(FEATURE_NAMES),positives=pos,negatives=n-pos,
              negative_budget_per_entity=budget,candidate_pairs=ncandidate,
              candidate_recall=recovered/ntrue if ntrue else None,
              all_true_links=ntrue,queries=observed_queries,zero_candidate_queries=empty_queries,
              positive_injection=False,feature_names=FEATURE_NAMES)
    write_json(path/'meta.json',info);write_json(Path(cfg['work_dir'])/'reports'/f'{label}_pairs.json',info)
    log(f'{label}: {n:,} training pairs, {pos:,} positives; candidate recall={info["candidate_recall"]}')
    return path

def cache_pool(path):
    info=json.loads((path/'meta.json').read_text());n=info['rows']
    X=np.memmap(path/'X.dat',dtype=np.float32,mode='r',shape=(n,info['features']))
    y=np.memmap(path/'y.dat',dtype=np.int8,mode='r',shape=(n,))
    w=np.memmap(path/'w.dat',dtype=np.float32,mode='r',shape=(n,))
    weights=np.asarray(w/w.mean(),np.float32)
    return Pool(X,label=y,weight=weights,feature_names=FEATURE_NAMES)

def fit_ensemble(train_cache,cfg,label,eval_cache=None,iterations=None):
    train=cache_pool(train_cache);valid=cache_pool(eval_cache) if eval_cache else None
    models=[];details=[]
    use_gpu=cfg['task_type'].upper()=='GPU'
    if use_gpu:
        try:count=get_gpu_device_count()
        except Exception:count=0
        if count==0:
            if not cfg['allow_cpu_fallback']:raise RuntimeError('No CUDA GPU. Enable T4 x2 in Kaggle Settings.')
            log('WARNING: no CUDA GPU detected; using the slower CPU fallback.');use_gpu=False
        elif any(int(x)>=count for x in cfg['devices'].split(':')):
            cfg=cfg|{'devices':':'.join(map(str,range(count)))}
            log(f'WARNING: requested GPU IDs unavailable; using devices={cfg["devices"]}')
    for i,seed in enumerate(cfg['seeds']):
        params=dict(loss_function='Logloss',eval_metric='Logloss',iterations=(iterations[i] if iterations else cfg['max_iterations']),
            depth=cfg['depth'],learning_rate=cfg['learning_rate'],l2_leaf_reg=cfg['l2_leaf_reg'],
            random_seed=seed,border_count=128,boosting_type='Plain',thread_count=cfg['threads'],
            allow_writing_files=False,verbose=100,task_type='GPU' if use_gpu else 'CPU')
        if use_gpu:params.update(devices=cfg['devices'],gpu_ram_part=cfg['gpu_ram_part'],data_partition='DocParallel')
        model=CatBoostClassifier(**params)
        kwargs=dict(eval_set=valid,use_best_model=True,early_stopping_rounds=cfg['early_stopping_rounds']) if valid else {}
        log(f'Training {label} seed={seed} task={params["task_type"]} devices={params.get("devices","CPU")}')
        try:model.fit(train,**kwargs)
        except CatBoostError as exc:
            message=str(exc).lower()
            if use_gpu and cfg['allow_cpu_fallback'] and any(x in message for x in ['cuda','gpu memory','driver version','nccl']):
                log(f'WARNING: GPU failed; CPU fallback. GPU error: {str(exc)[:400]}')
                params={k:v for k,v in params.items() if k not in ['devices','gpu_ram_part','data_partition']};params['task_type']='CPU'
                model=CatBoostClassifier(**params);model.fit(train,**kwargs)
            else:raise
        model.save_model(str(Path(cfg['work_dir'])/'models'/f'{label}_{i}.cbm'))
        models.append(model)
        details.append(dict(seed=seed,iterations=int(model.tree_count_),task_type=params['task_type'],devices=params.get('devices'),
                            numeric_parameter_upper_bound=int(model.tree_count_*(2**cfg['depth']+2*cfg['depth']+2))))
    write_json(Path(cfg['work_dir'])/'reports'/f'{label}_models.json',details)
    importance=np.mean([m.get_feature_importance() for m in models],axis=0)
    pd.DataFrame({'feature':FEATURE_NAMES,'importance':importance}).sort_values('importance',ascending=False).to_csv(Path(cfg['work_dir'])/'reports'/f'{label}_feature_importance.csv',index=False)
    del train,valid;gc.collect()
    return models,details

def load_models(cfg,label):
    models=[]
    for i in range(len(cfg['seeds'])):
        m=CatBoostClassifier();m.load_model(str(Path(cfg['work_dir'])/'models'/f'{label}_{i}.cbm'));models.append(m)
    return models

def predict(models,X,threads):
    if not len(X):return np.empty(0,np.float64)
    return sum(m.predict_proba(X,thread_count=threads)[:,1] for m in models)/len(models)

def evaluate(q,t,truth,cfg,vectors,models,thresholds,label):
    metrics=MetricAccumulator(thresholds);work=Path(cfg['work_dir'])
    detail=open(work/'reports'/f'{label}_entity_metrics.tsv','w',encoding='utf-8',newline='') if len(thresholds)==1 else None
    writer=csv.writer(detail,delimiter='\t',lineterminator='\n') if detail else None
    if writer:writer.writerow(['source1_entity_id','country','true_links','candidates','predicted_links','tp','fp','fn','f05','true_ids','predicted_ids'])
    try:
        for b,qi,ti,sim in candidate_batches(q,t,cfg,vectors):
            y=pair_labels(b,t,qi,ti,truth)
            X=build_features(b,t,qi,ti,sim,cfg['threads']);p=predict(models,X,cfg['threads'])
            starts=np.searchsorted(qi,np.arange(len(b)+1));target_ids=t.entity_id.to_numpy()
            for i,row in enumerate(b.itertuples(index=False)):
                sl=slice(starts[i],starts[i+1]);gt=truth[row.entity_id]
                n,tp,f=metrics.add(row.country,len(gt),y[sl],p[sl])
                if writer:
                    mids=target_ids[ti[sl]][p[sl]>=thresholds[0]]
                    writer.writerow([row.entity_id,row.country,len(gt),starts[i+1]-starts[i],int(n[0]),int(tp[0]),int(n[0]-tp[0]),len(gt)-int(tp[0]),float(f[0]),','.join(sorted(gt)),','.join(sorted(mids))])
    finally:
        if detail:detail.close()
    records=metrics.records();pd.DataFrame(records).to_csv(work/'reports'/f'{label}_threshold_curve.csv',index=False)
    # A central point on a tied optimum plateau is less brittle than its upper edge.
    best_score=max(r['macro_f05'] for r in records)
    tied=[r for r in records if abs(r['macro_f05']-best_score)<1e-12]
    precision=max(r['micro_precision'] for r in tied)
    tied=[r for r in tied if abs(r['micro_precision']-precision)<1e-12]
    best=tied[len(tied)//2]
    result={'selected':best,'by_country':{g.removeprefix('country:'):metrics.records(g)[int(np.argmin(np.abs(metrics.thresholds-best['threshold'])))] for g in metrics.groups if g!='ALL'},
            'comparison_count':len(q)*len(t),
            'reduction_ratio':1-best['candidate_pairs']/max(1,len(q)*len(t)),
            'metric_note':'Macro F0.5 over all S1, singletons included. Blocking misses count as false negatives. Pair accuracy covers candidates only.',
            'generalization_note':'The 30% holdout contains only countries in training. It cannot measure performance on France.'}
    write_json(work/'reports'/f'{label}_metrics.json',result)
    log(f'{label}: {json.dumps(best,indent=2)}')
    return result

def cleanup_cache(path,cfg):
    if not cfg['keep_training_cache']:shutil.rmtree(path)


## Cell 9 — Prediction, validation and submission packaging
This cell writes `src/submission.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/submission.py
"""Streaming outputs, strict full validation and reproducible submission ZIP."""
from pathlib import Path
from itertools import zip_longest
import csv,gc,importlib.metadata,json,platform,re,shutil,subprocess,sys,zipfile
import joblib
import numpy as np
from common import log,write_json,sha256
from data import read_split
from retrieval import candidate_batches
from features import build_features
from training import load_models,predict

def infer(cfg):
    work=Path(cfg['work_dir']);q,t=read_split(cfg['data_root'],'test')
    threshold=json.loads((work/'reports/calibration_metrics.json').read_text())['selected']['threshold']
    models=load_models(cfg,'final');vectors=joblib.load(work/'models/vectorizers.joblib')
    counts={};match_tmp=work/'output/matching_results.tsv.tmp';cand_tmp=work/'output/candidate_pairs.tsv.tmp'
    with match_tmp.open('w',encoding='utf-8',newline='') as fm,cand_tmp.open('w',encoding='utf-8',newline='') as fc:
        mw=csv.writer(fm,delimiter='\t',lineterminator='\n');cw=csv.writer(fc,delimiter='\t',lineterminator='\n')
        mw.writerow(['source1_entity_id','matched_entity_ids']);cw.writerow(['source1_entity_id','candidate_entity_ids'])
        target_ids=t.entity_id.to_numpy()
        for b,qi,ti,sim in candidate_batches(q,t,cfg,vectors):
            X=build_features(b,t,qi,ti,sim,cfg['threads']);p=predict(models,X,cfg['threads'])
            starts=np.searchsorted(qi,np.arange(len(b)+1))
            for i,row in enumerate(b.itertuples(index=False)):
                sl=slice(starts[i],starts[i+1]);cids=target_ids[ti[sl]];mids=cids[p[sl]>=threshold]
                # This is exactly the candidate set scored by BOTH ensemble models.
                cw.writerow([row.entity_id,','.join(sorted(cids))]);mw.writerow([row.entity_id,','.join(sorted(mids))])
                d=counts.setdefault(row.country,dict(entities=0,candidates=0,matches=0,empty=0))
                d['entities']+=1;d['candidates']+=len(cids);d['matches']+=len(mids);d['empty']+=int(len(mids)==0)
    match_tmp.replace(work/'output/matching_results.tsv');cand_tmp.replace(work/'output/candidate_pairs.tsv')
    write_json(work/'reports/test_prediction_summary.json',{'threshold':threshold,'by_country':counts,'total_source1':len(q),'scored':False})
    log('Prediction complete. Test accuracy is unknown because test labels are unavailable.')

def ids_from_source(path):
    ids=set()
    with open(path,encoding='utf-8',newline='') as f:
        for row in csv.DictReader(f,delimiter='\t'):
            sid=row['entity_id']
            if not sid or sid in ids:raise ValueError(f'Empty/duplicate source ID: {sid}')
            ids.add(sid)
    return ids

def strict_validate(root,matching,candidates):
    root=Path(root);required=ids_from_source(root/'dataset/test/test_source1.tsv')
    valid=ids_from_source(root/'dataset/test/test_source2.tsv')|ids_from_source(root/'dataset/test/test_source3.tsv')
    seen=set();nlinks=ncands=0
    with open(matching,encoding='utf-8',newline='') as fm,open(candidates,encoding='utf-8',newline='') as fc:
        rm=csv.reader(fm,delimiter='\t');rc=csv.reader(fc,delimiter='\t')
        assert next(rm)==['source1_entity_id','matched_entity_ids'],'Incorrect matching header'
        assert next(rc)==['source1_entity_id','candidate_entity_ids'],'Incorrect candidate header'
        for number,(m,c) in enumerate(zip_longest(rm,rc),2):
            if m is None or c is None or len(m)!=2 or len(c)!=2:raise ValueError(f'Malformed/mismatched row {number}')
            sid=m[0]
            if sid!=c[0] or sid not in required or sid in seen:raise ValueError(f'Invalid S1 row {number}: {sid}')
            seen.add(sid)
            ml=m[1].split(',') if m[1] else [];cl=c[1].split(',') if c[1] else []
            ms,cs=set(ml),set(cl)
            if len(ms)!=len(ml) or len(cs)!=len(cl):raise ValueError(f'Duplicate list ID in {sid}')
            if not ms<=cs:raise ValueError(f'Final match absent from candidates in {sid}')
            if not cs<=valid or any(not x.startswith(('S2-','S3-')) for x in cs):raise ValueError(f'Unknown candidate ID in {sid}')
            nlinks+=len(ms);ncands+=len(cs)
    if seen!=required:raise ValueError(f'Missing {len(required-seen)} S1 rows')
    return dict(status='PASS',entities=len(seen),matching_links=nlinks,candidate_links=ncands,
                all_ids_checked=True,subset_check='hard failure',candidate_memory='one row at a time')

def documentation(cfg,work):
    audit=json.loads((work/'reports/data_audit.json').read_text())
    cal=json.loads((work/'reports/calibration_metrics.json').read_text())
    hold=json.loads((work/'reports/holdout_metrics.json').read_text())
    summary=json.loads((work/'reports/test_prediction_summary.json').read_text())
    models=json.loads((work/'reports/final_models.json').read_text())
    country_table='\n'.join(f'| {c or "Unknown"} | {v["entities"]} | {v["macro_f05"]:.6f} | {v["entity_exact_match_accuracy"]:.6f} |' for c,v in hold['by_country'].items())
    main=hold['selected']
    return f'''# Amazon ML Challenge 2026 — methodology

## Team
{cfg['team_name']}

## Methodology used
Match every S1 record to zero or more S2/S3 records. Preserve all confidently predicted matches; do not impose one-to-one assignment. This submission trains a numerical pair classifier from the supplied data. No business lookup, geocoding, downloaded business data, pretrained model or remote inference API is used.

## Validation design
The outer split is {cfg['holdout_fraction']:.0%} holdout by complete entity groups. Realized row fractions: training {audit['actual_outer_train_fraction']:.4%}, holdout {audit['actual_outer_holdout_fraction']:.4%}. Groups join shared ground-truth target IDs and identical nonempty normalized business/address/country signatures. Linked S2/S3 rows follow their S1 group; unlinked rows are assigned reproducibly by signature hash. Inside the 70% training side, {cfg['calibration_fraction_of_training']:.0%} is used for early stopping and threshold calibration. TF-IDF is fitted exclusively on the inner fit side and frozen for all later stages. The model is then refitted on all 70% and evaluated once on the 30% holdout with fixed iterations and threshold. Final submission models are refitted on 100% of training labels.

## Candidate generation / blocking strategy
Normalize Unicode case and accents while retaining Unicode letters. Keep both full names and suffix-reduced names. Normalize a small handwritten list of address abbreviations; retain numbers. Search S2 and S3 separately. Country blocking: {cfg['use_country_block']}; enabled automatically only when inner-fit labeled links have no known cross-country mismatch. Any country string, including France, is accepted. Blank query countries search all targets; known query countries also search blank-country targets. Country consistency is an assumption, not a guarantee for unseen data.

Union top-k name character TF-IDF ({cfg['name_topk']}), address character TF-IDF ({cfg['address_topk']}), joint word TF-IDF ({cfg['word_topk']}) and name character hashing ({cfg['hash_topk']}) results per source. Add every exact normalized full-name/address signature match. Sparse cosine floor {cfg['similarity_floor']}; no dense all-pairs matrix. candidate_pairs.tsv is exactly the union actually scored by the final models. Training labels are never injected into candidate sets.

## Model architecture and feature engineering
Equal-probability ensemble of {len(cfg['seeds'])} CatBoost binary classifiers, depth {cfg['depth']}, learning rate {cfg['learning_rate']}, L2 {cfg['l2_leaf_reg']}. Features include four cosine similarities, full/core name and address edit similarities, token order/set similarities, token overlap/containment, missingness, length ratios, acronym relationships, street-number and long-number agreement/conflicts, country agreement/missingness and source indicator. No entity ID or numeric ID suffix is fed to the model. All recovered positive pairs are retained; bounded hard negatives plus random retrieved negatives control training size. Per-query weights prevent large candidate lists from dominating.

GPU training requests {cfg['devices']}; actual training configuration is recorded in reports/final_models.json. GPU reductions may be nondeterministic. Final iterations and estimated numerical tree-parameter upper bounds: {json.dumps(models)}. CatBoost is Apache-2.0; authored pipeline and resulting models are released under MIT in this package. Total tree parameters are far below 8 billion.

## Experiments and measured results
Only results actually generated by this run are reported. The calibration sweep selected threshold {cal['selected']['threshold']}; this threshold was frozen before outer holdout evaluation. Models, split manifest, candidate-recall diagnostics, threshold curve, feature importance, data hashes and per-entity holdout errors are included in reports/.

| Metric on untouched 30% holdout | Value |
|---|---:|
| Macro F0.5 (official objective) | {main['macro_f05']:.6f} |
| Exact complete-match-set accuracy | {main['entity_exact_match_accuracy']:.6f} |
| Micro precision | {main['micro_precision']:.6f} |
| Micro recall | {main['micro_recall']:.6f} |
| Candidate link recall | {main['candidate_recall']} |
| Singleton accuracy | {main['singleton_accuracy']} |

| Country | S1 entities | Macro F0.5 | Exact-set accuracy |
|---|---:|---:|---:|
{country_table}

## Conclusion and limitations
This is one measured pipeline, not a claimed winning solution. A training-derived holdout cannot establish accuracy in unseen France. Separate holdout target pools contain fewer distractors than the complete test universe. Country blocking, normalization and top-k retrieval can omit valid matches; blocking recall explicitly includes such misses. GPU speed does not remove the CPU cost of large sparse retrieval and string feature computation. Test labels are unknown, so no test accuracy is claimed.

## Final outputs
Test S1 rows: {summary['total_source1']}. One matching row and one candidate row per S1; empty strings represent no match. Full strict streaming validation checks both files, all target IDs, duplicates and matched-subset-of-candidates. The provided validator is also run. All rows, including unseen countries, are included.

## Reproduction
See code/business_entity_resolution/README.md. The package contains exact fitted model artifacts for inference, source code for full retraining, run configuration, pinned used dependencies and environment metadata. No input dataset copy is bundled.
'''

def package(cfg):
    work=Path(cfg['work_dir']);root=Path(cfg['data_root'])
    result=strict_validate(root,work/'output/matching_results.tsv',work/'output/candidate_pairs.tsv')
    write_json(work/'reports/strict_validation.json',result);log(f'Strict full validation: {result}')
    empty=work/'artifacts/validator_workdir';empty.mkdir(exist_ok=True)
    cmd=[sys.executable,str(work/'artifacts/official_validate_submission.py'),
         '--matching',str(work/'output/matching_results.tsv'),'--test-dir',str(root/'dataset/test'),'--check-ids']
    if cfg['run_official_full_candidate_check']:cmd+=['--candidate',str(work/'output/candidate_pairs.tsv')]
    # The official script stores all candidates in RAM. Default: run it on matching only;
    # our strict streaming validator above checks the complete candidate file and IDs.
    completed=subprocess.run(cmd,cwd=empty,text=True,capture_output=True)
    (work/'reports/official_validator.log').write_text(completed.stdout+'\n'+completed.stderr)
    print(completed.stdout,flush=True)
    if completed.returncode:raise RuntimeError('Provided validator failed; inspect reports/official_validator.log')
    modules=['numpy','pandas','scipy','scikit-learn','catboost','rapidfuzz','sparse-dot-topn','joblib']
    requirements='\n'.join(f'{m}=={importlib.metadata.version(m)}' for m in modules)+'\n'
    (work/'requirements.lock.txt').write_text(requirements)
    driver=subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout if shutil.which('nvidia-smi') else 'No NVIDIA GPU'
    write_json(work/'reports/runtime.json',{'python':sys.version,'platform':platform.platform(),'gpu':driver,'dependencies':requirements.splitlines()})
    team=re.sub(r'[^A-Za-z0-9_-]+','_',cfg['team_name']) or 'team'
    destination=work/f'{team}_submission.zip'
    prefix='code/business_entity_resolution/'
    doc=documentation(cfg,work)
    (work/'Documentation_template.md').write_text(doc,encoding='utf-8')
    with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
        for filename in ['matching_results.tsv','candidate_pairs.tsv']:
            z.write(work/'output'/filename,'output/'+filename)
        z.writestr('Documentation_template.md',doc)
        for folder in ['src','reports']:
            for p in sorted((work/folder).rglob('*')):
                if p.is_file() and '__pycache__' not in p.parts:z.write(p,prefix+str(p.relative_to(work)))
        for p in [work/'models/vectorizers.joblib',*sorted((work/'models').glob('final_*.cbm'))]:
            z.write(p,prefix+str(p.relative_to(work)))
        for name in ['README.md','LICENSE','THIRD_PARTY.md']:
            if not (work/name).exists():raise FileNotFoundError(f'Missing reproducibility file: {name}')
            z.write(work/name,prefix+name)
        z.writestr(prefix+'requirements.txt',requirements)
        z.writestr(prefix+'run_config.json',json.dumps(cfg,indent=2))
        z.write(work/'artifacts/resolved_config.json',prefix+'artifacts/resolved_config.json')
        z.write(work/'artifacts/official_validate_submission.py',prefix+'artifacts/official_validate_submission.py')
        if (work/'artifacts/provided_Documentation_template.md').exists():
            z.write(work/'artifacts/provided_Documentation_template.md','Documentation_template.provided.md')
    write_json(work/'reports/output_hashes.json',{'matching_sha256':sha256(work/'output/matching_results.tsv'),
        'candidate_sha256':sha256(work/'output/candidate_pairs.tsv'),'zip_sha256':sha256(destination)})
    log(f'Package ready: {destination} ({destination.stat().st_size/1024**2:.1f} MiB)')


## Cell 10 — Stage runner
This cell writes `src/main.py`; execution happens after all modules have been created.

In [ ]:
%%writefile /kaggle/working/amazon_er/src/main.py
"""Stage CLI. Each notebook run cell starts a fresh process to release memory."""
from pathlib import Path
import argparse,gc,json,subprocess,sys,time
import joblib
from common import config,log,write_json
from data import prepare,load_training
from retrieval import fit_vectorizers
from training import build_cache,fit_ensemble,evaluate,cleanup_cache
from submission import infer,package

STAGES=['prepare','tune','holdout','final','predict','package']

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--config',default='run_config.json');p.add_argument('--stage',choices=STAGES+['all','infer-saved'],required=True)
    p.add_argument('--data-root');p.add_argument('--work-dir');args=p.parse_args()
    overrides={}
    if args.data_root:overrides['data_root']=str(Path(args.data_root).resolve())
    if args.work_dir:overrides['work_dir']=str(Path(args.work_dir).resolve())
    cfg=config(args.config,overrides)
    work=Path(cfg['work_dir'])
    for d in ['artifacts','cache','models','reports','output']:(work/d).mkdir(parents=True,exist_ok=True)
    if args.stage=='all':
        for stage in STAGES:
            command=[sys.executable,'-u',__file__,'--config',str(Path(args.config).resolve()),'--stage',stage]
            if args.data_root:command+=['--data-root',args.data_root]
            if args.work_dir:command+=['--work-dir',args.work_dir]
            subprocess.run(command,check=True)
        return
    started=time.time();state_path=work/'artifacts/stage_status.json'
    state=json.loads(state_path.read_text()) if state_path.exists() else {}
    if args.stage in STAGES:
        current=STAGES.index(args.stage)
        if current and not state.get(STAGES[current-1],{}).get('complete'):
            raise RuntimeError(f'Run stage {STAGES[current-1]} first. Cells must run in order.')
        for later in STAGES[current:]:state.pop(later,None)
        write_json(state_path,state)
    if args.stage=='prepare':
        prepare(cfg)
        cfg=json.loads((work/'artifacts/resolved_config.json').read_text())
        q,t,_=load_training(cfg,'fit');fit_vectorizers(q,t,cfg)
    else:
        saved=json.loads((work/'artifacts/resolved_config.json').read_text())
        if args.stage!='infer-saved':
            changes=[k for k,v in cfg.items() if k not in ['data_root','work_dir','use_country_block'] and saved.get(k)!=v]
            if changes:raise RuntimeError(f'Configuration changed ({changes}); rerun prepare and subsequent stages.')
        cfg=saved|{'work_dir':str(work),'data_root':cfg['data_root'] if args.data_root else saved['data_root']}
        vectors=joblib.load(work/'models/vectorizers.joblib') if args.stage in ['tune','holdout','final'] else None
        if args.stage=='tune':
            q,t,gt=load_training(cfg,'fit');fit_cache=build_cache(q,t,gt,cfg,vectors,'inner_fit');del q,t,gt;gc.collect()
            q,t,gt=load_training(cfg,'calibration');cal_cache=build_cache(q,t,gt,cfg,vectors,'inner_calibration')
            models,details=fit_ensemble(fit_cache,cfg,'inner',eval_cache=cal_cache)
            cleanup_cache(fit_cache,cfg);cleanup_cache(cal_cache,cfg)
            evaluate(q,t,gt,cfg,vectors,models,cfg['threshold_grid'],'calibration')
            write_json(work/'artifacts/selected_iterations.json',[x['iterations'] for x in details])
        elif args.stage in ['holdout','final']:
            subset='development' if args.stage=='holdout' else 'all'
            q,t,gt=load_training(cfg,subset);cache=build_cache(q,t,gt,cfg,vectors,subset);del q,t,gt;gc.collect()
            rounds=json.loads((work/'artifacts/selected_iterations.json').read_text())
            label='development' if args.stage=='holdout' else 'final'
            models,_=fit_ensemble(cache,cfg,label,iterations=rounds);cleanup_cache(cache,cfg)
            if args.stage=='holdout':
                threshold=json.loads((work/'reports/calibration_metrics.json').read_text())['selected']['threshold']
                q,t,gt=load_training(cfg,'holdout')
                evaluate(q,t,gt,cfg,vectors,models,[threshold],'holdout')
        elif args.stage in ['predict','infer-saved']:infer(cfg)
        elif args.stage=='package':package(cfg)
    if args.stage!='infer-saved':
        state[args.stage]={'complete':True,'seconds':round(time.time()-started,2)};write_json(state_path,state)
    log(f'Completed {args.stage} in {(time.time()-started)/60:.1f} minutes')

if __name__=='__main__':main()


## Cell 11 — Reproduction documents and syntax check

In [ ]:
from pathlib import Path
import compileall
BASE=Path('/kaggle/working/amazon_er')
(BASE/'README.md').write_text("# Amazon business entity resolution — Kaggle pipeline\n\nThis pipeline has not been benchmarked on your actual challenge dataset by its author. The notebook measures its own results when run in Kaggle. Synthetic tests establish software behavior, not challenge accuracy.\n\n## Kaggle: import and run\n\n1. Import `Amazon_ML_Kaggle.ipynb` using Kaggle **File → Import Notebook**.\n2. Attach the challenge dataset and enable **GPU T4 x2**. The configuration uses `/kaggle/input/datasets/mayankr293/aws-ml-challenge-dataset/student_resource`; automatic discovery handles the alternate Kaggle mount layout if there is exactly one resource directory.\n3. Internet is needed only if the three pinned add-on packages are missing. No business data or model weights are downloaded. With internet disabled, attach wheels in a `wheels` directory and configure the setup cell's `WHEEL_DIR`.\n4. Change `TEAM_NAME` if desired. Execute cells top to bottom, or Save Version → Save & Run All. Each Python module is created with `%%writefile`; each stage is a checked subprocess and releases its RAM when it finishes.\n5. Read `reports/holdout_metrics.json`. The official objective is `selected.macro_f05`; `entity_exact_match_accuracy` is the percentage of complete S1 match sets predicted exactly. `candidate_pair_accuracy` is only a diagnostic and can be misleading under imbalance.\n6. Upload `/kaggle/working/amazon_er/output/matching_results.tsv` to the leaderboard. Keep the submission ZIP and both output TSVs for the final submission.\n\n## Architecture\n\n`common.py`: configuration and normalization.\n`data.py`: schema/ID checks, entity grouping, inner split and outer 70/30 split, provenance audit.\n`retrieval.py`: TF-IDF/name hashing, source/country partitions, sparse top-k union and exact signature lookup.\n`features.py`: 50 numerical pair features.\n`metrics.py`: macro F0.5, exact-set accuracy, singleton and blocking metrics.\n`training.py`: bounded hard-negative sampling, disk-backed feature cache, CatBoost ensemble, threshold sweep and holdout scoring.\n`submission.py`: streamed prediction, strict validation and final ZIP.\n`main.py`: stage sequencing and saved-model inference.\n\nThe actual feature count is saved with every feature cache and can be inspected from `src/features.py` (`FEATURE_NAMES`).\n\n## Exact validation procedure\n\n- The unit of the outer split is a group of real-entity records, not a random pair. Shared target IDs and identical nonempty normalized business/address/country signatures stay together. Approximately 70% of S1 rows train the model and 30% evaluate it; full group preservation can slightly change row percentages.\n- Within the 70%, 80% is inner fit and 20% is calibration (roughly 56%/14%/30% of S1 overall). Each linked S2/S3 record follows its S1 group. Unlinked rows are assigned deterministically by normalized signature.\n- TF-IDF statistics are fitted only on inner-fit records, with a configurable 180,000-record cap for this unsupervised fitting step. Classifier development still uses all entities in its designated partition. Hashing handles unseen vocabulary.\n- Two seed models train on inner-fit pairs; early stopping and threshold selection use internal calibration only. Calibration scoring uses the full generated candidate set, including all singleton queries. No true positives are added to retrieval by using labels.\n- New models are fitted on all 70% using the chosen iterations; fixed threshold evaluation on the untouched 30% gives the reported result. Final submission models then train on all 100% of labeled entities, with the same frozen representation and threshold.\n- After inspecting holdout errors, any further design tuning makes this holdout a development set. Do not label repeated tuned results as untouched validation.\n- Validation does not contain France if France is absent from the supplied training data. Separate holdout candidate pools contain fewer distractors than the full test pool.\n\n## Retrieval and matching choices\n\nFour candidate branches search S2 and S3 separately: name character TF-IDF top 16, address character TF-IDF top 10, combined word TF-IDF top 12, and name character hashing top 6. Results are unioned; exact normalized full-name/address matches retain all copies. There is no arbitrary final cap or one-to-one assignment. The sparse similarity floor is 0.05; it is configurable and contributes to blocking recall.\n\nCountry blocking is enabled automatically only if inner-fit positive links do not contradict it. Unseen labels are allowed; blank query countries search every country. This assumes consistent country labels in unseen data. Set `country_block=false` and rerun all stages to disable it. This is a tradeoff between candidate cost and cross-country recall.\n\nAll recovered training positives plus bounded hard and random retrieved negatives train the classifier. Negative counts adapt to a four-million-pair budget; no S1 query is discarded to satisfy this budget. Queries with no retrieved pairs cannot supply pair-classifier examples and remain in all evaluation denominators. No ID or country vocabulary enters the model; only country agreement/missingness is a feature. Pair weights are equalized by query. Two CatBoost models' probabilities are averaged, then a single calibrated threshold chooses matches. All accepted S2/S3 IDs are preserved.\n\nEvery candidate in the exported candidate file is scored by both final classifiers. Empty candidate and matching rows are retained. Test IDs are never guessed or generated.\n\n## GPU and memory\n\nCatBoost uses native multi-GPU training with `task_type=GPU`, `devices=0:1`, `data_partition=DocParallel`. This does not require PyTorch or DDP. Seed models are trained sequentially using both GPUs. Sparse retrieval, text normalization and string features use CPU threads; those stages will not keep the GPUs busy. Training features are float32 disk-backed arrays, candidate prediction is batched, and no dense S1×target matrix is constructed. CatBoost still creates its own training Pool in RAM. GPU errors can trigger an explicitly logged CPU fallback.\n\nFor lower memory, reduce `query_batch_size` and `transform_batch_size`; lower `max_train_pairs` to reduce classifier memory, provided the budget still covers positive pairs and at least one negative per entity. Reducing candidate top-k changes recall; always inspect candidate recall. Actual runtime and peak memory depend on unseen dataset size and text lengths; there is no promised runtime. GPU training can vary slightly between identical seeded runs.\n\n## Run from source\n\nFrom this directory, install requirements and run:\n\n```bash\npython -m pip install -r requirements.txt\npython -u src/main.py --config run_config.json --stage all --data-root /path/to/student_resource --work-dir /absolute/path/to/this_directory\n```\n\nRun stages separately with `--stage prepare`, `tune`, `holdout`, `final`, `predict`, `package`. Earlier stages invalidate completion markers for subsequent stages. If you change configuration, rerun `prepare` and all later stages.\n\nThe final submission archive includes fitted model files. From its `code/business_entity_resolution` directory, reproduce inference without retraining:\n\n```bash\npython -u src/main.py --config run_config.json --stage infer-saved --data-root /path/to/student_resource --work-dir /absolute/path/to/code/business_entity_resolution\n```\n\nThe saved vectorizers, final CatBoost models and calibration threshold determine predictions. CPU prediction is used for stable, inexpensive tree inference. Outputs are placed in that directory's `output/`.\n\n## Outputs\n\n- `output/matching_results.tsv`: only leaderboard upload.\n- `output/candidate_pairs.tsv`: exact candidate set scored during final inference.\n- `models/final_0.cbm`, `models/final_1.cbm`, `models/vectorizers.joblib`: trained artifacts.\n- `reports/holdout_metrics.json`: fixed-threshold 30% evaluation.\n- `reports/holdout_entity_metrics.tsv`: errors with truth/prediction IDs, including blocking misses.\n- `reports/calibration_threshold_curve.csv`: internal threshold search, never the outer holdout.\n- `reports/entity_split.tsv`, `reports/data_audit.json`: split and input provenance.\n- `reports/test_prediction_summary.json`: output counts by country; no test accuracy claim.\n- `Documentation_template.md`: populated methodology and measured results. The provided template is preserved separately; review organizer-specific headings and add your actual team identity before final upload.\n- `<team_name>_submission.zip`: code, fitted artifacts, both outputs, reports, methodology and pinned used dependencies. Input datasets and temporary pair caches are excluded.\n\nStrict streaming validation checks both files and all IDs, rejecting even matched-subset-of-candidate violations. The supplied validator also runs with `--check-ids` on matching results. By default its candidate pass is skipped to avoid loading the entire candidate graph into RAM; the strict validator already performs the full candidate checks. The official log may therefore contain its expected candidate-skip warning. Set `run_official_full_candidate_check=true` if sufficient RAM is available.\n\nNo external business lookup, geocoding, remote model call or external training data is used. No score, accuracy, deadline completion or winning position is guaranteed.\n",encoding='utf-8')
(BASE/'LICENSE').write_text('MIT License\n\nCopyright (c) 2026 Pipeline contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n',encoding='utf-8')
(BASE/'THIRD_PARTY.md').write_text("# Licenses and execution notes\n\n- CatBoost 1.2.8: Apache-2.0, https://github.com/catboost/catboost/blob/v1.2.8/LICENSE\n- RapidFuzz 3.13.0: MIT, https://github.com/rapidfuzz/RapidFuzz/blob/v3.13.0/LICENSE\n- sparse-dot-topn 1.1.5: Apache-2.0, https://github.com/ing-bank/sparse_dot_topn\n- NumPy, SciPy, pandas, scikit-learn and joblib are supporting libraries; their upstream licenses apply. Their exact used versions are captured in the final package.\n- No pretrained neural model, external business data, remote inference or geocoding service is used. CatBoost classifiers are trained only on the supplied labels, have far fewer than 8 billion numerical tree parameters, and are saved with the authored pipeline under the MIT license in this package.\n- GPU training settings: https://catboost.ai/docs/en/features/training-on-gpu\n- Sparse multiplication API: https://github.com/ing-bank/sparse_dot_topn\n- The supplied organizer validator retains its original authorship; it is copied from the user's dataset and used unmodified at runtime.\n\nThe local synthetic verification covers CPU execution. Real dual-T4 execution and competition accuracy must be established by the notebook run in Kaggle.\n",encoding='utf-8')
(BASE/'requirements.txt').write_text('catboost==1.2.8\nrapidfuzz==3.13.0\nsparse-dot-topn==1.1.5\nnumpy>=1.24,<3\npandas>=1.5,<3\nscipy>=1.10,<2\nscikit-learn>=1.2,<2\njoblib>=1.2,<2\n',encoding='utf-8')
assert compileall.compile_dir(str(BASE/'src'),quiet=1), 'A Python module has a syntax error'
print('All modules written and syntax-checked.')


## Cell 12 — Audit data and create isolated partitions
Reads real input schemas, IDs, countries and row counts; writes provenance and split manifest; fits frozen text representations only on the inner-fit portion.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','prepare'],cwd=BASE,check=True)


## Cell 13 — Train internally and calibrate
Uses only the 70% training side. Builds bounded training pairs, selects early-stopped iterations, then selects the threshold from full internal-calibration candidate predictions.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','tune'],cwd=BASE,check=True)


## Cell 14 — Train on 70% and evaluate the untouched 30%
Fits new models on all 70% using the fixed iteration counts. The threshold remains frozen. The resulting score is your independent validation result.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','holdout'],cwd=BASE,check=True)


### Inspect the measured result
Macro F0.5 is the competition objective. Do not confuse it with ordinary binary accuracy. Once this holdout is used to select new designs, subsequent tuned results are no longer an untouched estimate.

In [ ]:
from pathlib import Path
import json,pandas as pd
BASE=Path('/kaggle/working/amazon_er')
report=json.loads((BASE/'reports/holdout_metrics.json').read_text())
r=report['selected']
print(f"Macro F0.5: {r['macro_f05']:.6f}")
print(f"Exact complete-match-set accuracy: {100*r['entity_exact_match_accuracy']:.2f}%")
print(f"Precision: {100*r['micro_precision']:.2f}% | Recall: {100*r['micro_recall']:.2f}%")
print('Candidate recall:',r['candidate_recall'],'| Singleton accuracy:',r['singleton_accuracy'])
print('Frozen threshold:',r['threshold'])
display(pd.DataFrame(report['by_country']).T)
print('Real test accuracy is unavailable: there are no test labels.')


## Cell 15 — Train the submission models on all labeled data
After recording holdout evaluation, train the final two classifiers on 100% of supplied training labels. Keep the selected iterations, representation and threshold.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','final'],cwd=BASE,check=True)


## Cell 16 — Predict all test entities
Includes France and every other country label. Writes the exact scored candidate set and final matching set in streaming batches.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','predict'],cwd=BASE,check=True)


## Cell 17 — Validate and build the final ZIP
Runs strict streaming checks on both files and the supplied validator on matching results with ID checking. The supplied validator candidate-skip warning is expected in memory-safe mode. Writes actual results into the methodology and packages the submission.

In [ ]:
from pathlib import Path
import subprocess,sys
BASE=Path('/kaggle/working/amazon_er')
subprocess.run([sys.executable,'-u',str(BASE/'src/main.py'),'--config',str(BASE/'run_config.json'),'--stage','package'],cwd=BASE,check=True)


## Download outputs
Upload **only `matching_results.tsv`** to the live leaderboard. Keep the ZIP for final submission. Review the generated methodology against organizer-specific fields and enter the actual team identity before final upload. Training time and accuracy depend on the real data; no deadline completion or winning score is promised.

In [ ]:
from pathlib import Path
from IPython.display import display,FileLink
BASE=Path('/kaggle/working/amazon_er')
for path in [BASE/'output/matching_results.tsv',BASE/'output/candidate_pairs.tsv',BASE/'reports/holdout_metrics.json',BASE/'Documentation_template.md',*BASE.glob('*_submission.zip')]:
    print(f'{path.name}: {path.stat().st_size/1024**2:.2f} MiB')
    display(FileLink(str(path)))
